In [1]:
# import newkernelo as newker
import xllim as newker
import kernelo as oldker
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)

## Global parameters

In [2]:
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 50
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 300
train_ratio_ll = 1e-5
train_floor = 1e-12

##### Generate random dataset

In [3]:
L, D, K, N = 4, 9, 5, 1000

x_gen = np.random.rand(N,L)
y_gen = np.random.rand(N,D)

#### Generate sobol Test Model dataset

In [4]:
L, D, K, N = 4, 9, 5, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 4)
(100, 9)


#### Generate sobol RPV model dataset

In [5]:
L, D, K, N = 3, 71, 10, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", "../../../pytest/models").create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 3)
(100, 71)


## OLD GLLiM

In [6]:
# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=oldker.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=oldker.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_old= oldker.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)


In [7]:

print("Initializing GLLIM model")
gllim_old.initialize(x_gen, y_gen)
gllim_old_params_0 = gllim_old.exportModel() # you can export your gllim model parameters

print("Training model")
gllim_old.train(x_gen, y_gen)
gllim_old_params = gllim_old.exportModel() # you can export your gllim model parameters


INFO:root:Start Multi initialization
INFO:root:Initialisation : 1
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 108.204364, Best log likelihood : 108.204364


Initializing GLLIM model


INFO:root:Initialisation : 2
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 102.311793, Best log likelihood : 108.204364
INFO:root:Initialisation : 3
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 138.828032, Best log likelihood : 138.828032
INFO:root:Initialisation : 4
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 107.922886, Best log likelihood : 138.828032
INFO:root:Initialisation : 5
INFO:root:	Generate GMM me

Training model


### Get theta_0 from oldker

In [8]:
# theta_0 = newker.GLLiMParameters(L,D,K, "full", "diag")

# print(np.array(gllim_parameters_0.Pi).shape)
# print(np.array(gllim_parameters_0.A).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.array(gllim_parameters_0.Gamma).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

# theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
# theta_0.A = np.copy(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)))
# theta_0.C = np.copy(np.array(gllim_parameters_0.C))
# # theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
# theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
# theta_0.B = np.copy(np.array(gllim_parameters_0.B))
# # theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
# Sigma_diag = np.zeros((K,D))
# for k in range(K):
#     Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
# theta_0.Sigma = Sigma_diag

# print(np.sum(theta_0.Pi))
# gllim.setParams(theta_0)

## NEW GLLiM

In [9]:
n_hidden_variables = 0
L = L + n_hidden_variables
new_gllim = newker.GLLiM(K,D,L,gamma_type.lower(), sigma_type.lower(), n_hidden_variables)

verbose = 1
X_gen = x_gen.T
Y_gen = y_gen.T

new_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
new_gllim_params_0 = new_gllim.getParams()

new_gllim.train(X_gen, Y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose)
new_gllim_params = new_gllim.getParams()


INFO: GLLiM Parameters initialized
INFO: GLLiM dimensions are (L=3, D=71, K=10)
INFO: GLLiM constraints are :
	gamma_type = 'full',
	sigma_type = 'diag'.
INFO: Start Initialization
INFO: Initialisation : 1
INFO: 	Generate GMM means
INFO: 	Generate GMM covariance matrices
INFO: 	Train the GMM model
INFO: 	Compute Initial theta vector of the GLLiM model from GMM
INFO: 	Train the initial GLLiM model
INFO: Start GLLiM-EM Training
INFO: Iteration : 1, log likelihood : 1.306648
INFO: Iteration : 2, log likelihood : 1.901673
INFO: Iteration : 3, log likelihood : 2.006416
INFO: Iteration : 4, log likelihood : 2.006418
INFO: Iteration : 5, log likelihood : 2.006418
INFO: Iteration : 6, log likelihood : 2.006418
INFO: Iteration : 7, log likelihood : 2.006418
INFO: Iteration : 8, log likelihood : 2.006418
INFO: Iteration : 9, log likelihood : 2.006418
INFO: Iteration : 10, log likelihood : 2.006418
INFO: Iteration : 11, log likelihood : 2.006418
INFO: Iteration : 12, log likelihood : 2.006418
INF

likelihood : -inf
INFO: Iteration : 24, log likelihood : -inf
INFO: Iteration : 25, log likelihood : -inf
INFO: Iteration : 26, log likelihood : -inf
INFO: Iteration : 27, log likelihood : -inf
INFO: Iteration : 28, log likelihood : -inf
INFO: Iteration : 29, log likelihood : -inf
INFO: Iteration : 30, log likelihood : -inf
INFO: Iteration : 31, log likelihood : -inf
INFO: Iteration : 32, log likelihood : -inf
INFO: Iteration : 33, log likelihood : -inf
INFO: Iteration : 34, log likelihood : -inf
INFO: Iteration : 35, log likelihood : -inf
INFO: Iteration : 36, log likelihood : -inf
INFO: Iteration : 37, log likelihood : -inf
INFO: Iteration : 38, log likelihood : -inf
INFO: Iteration : 39, log likelihood : -inf
INFO: Iteration : 40, log likelihood : -inf
INFO: Iteration : 41, log likelihood : -inf
INFO: Iteration : 42, log likelihood : -inf
INFO: Iteration : 43, log likelihood : -inf
INFO: Iteration : 44, log likelihood : -inf
INFO: Iteration : 45, log likelihood : -inf
INFO: Iteratio

In [10]:
print(gllim_old_params_0.Pi)
print(new_gllim_params_0.Pi)

print("\n")
print(gllim_old_params.Pi)
print(new_gllim_params.Pi)

print("\n")
print(gllim_old_params.B)
print(new_gllim_params.B)

[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.1  0.11 0.07 0.08 0.1  0.1  0.1  0.22 0.08 0.04]]


[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.1  0.11 0.07 0.08 0.1  0.1  0.1  0.22 0.08 0.04]]


[[ 5.53464328e-01  5.61395258e-01  6.32268989e-01  7.60396978e-01
   9.51415234e-01  1.20077834e+00  1.46498454e+00  1.46497963e+00
   1.20081137e+00  9.51255542e-01]
 [ 7.60461805e-01  6.32260235e-01  5.61394930e-01  5.53363415e-01
   1.46127591e-01  2.18296391e-01  2.85246072e-01  3.66719026e-01
   4.79034983e-01  6.41727550e-01]
 [ 8.77550701e-01  1.20095428e+00  1.58787270e+00  1.87371829e+00
   1.67103889e+00  1.46498217e+00  1.33912253e+00  1.36972847e+00
   3.82746716e+00  3.57479282e+00]
 [ 3.54327360e+00  2.43279943e+00  1.67115852e+00  1.12416022e+00
   7.60413008e-01  5.24276906e-01  3.67009945e-01  2.53434989e-01
   1.60043115e-01  6.46405297e-02]
 [-6.52129289e-02 -3.13529558e-01 -1.72263288e+00 -7.04285972e-01
  -2.87113051e-01 -6.51694910e-02  8.56722170e-02 

In [11]:
# Y_gen = np.array(y_gen[:5].T)
# pred = new_gllim.inverseDensities(Y_gen)
# print(pred.fullGMM.mean)
# print(x_gen[:5])
# print("\n")


predicator = oldker.PredictionConfig(2, 2, 1e-10, gllim_old).create()
for i in range(5):
    print(x_gen[i])

    prediction = predicator.predict(y_gen[i], np.zeros(D))
    print(prediction.meansPred.mean)

    pred = new_gllim.inverseDensities(y_gen[i])
    print(pred.fullGMM.mean)
    print("\n")


[0.5 0.5 0.5]
[0.62569057 0.57057325 0.49221936]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[[0.56845151 0.53920169 0.51749624]]


[0.75 0.25 0.25]
[0.75265593 0.20447708 0.23439607]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[[0.75654118 0.21240258 0.23605104]]


[0.25 0.75 0.75]
[0.84870943 0.75358438 0.89956052]
[[0.06734091 0.30511374 0.61953894]]


[0.375 0.375 0.625]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[0.3647309  0.3587202  0.62201659]
[[0.3359536  0.29307657 0.61188607]]


[0.875 0.

In [12]:
pred.mergedGMM.means

array([], shape=(0, 0), dtype=float64)

In [13]:
insights = new_gllim.getInsights()

In [14]:
print(insights.time)
print(type(insights.time))
print(type(insights.initialisation.start_time))

0:00:01
<class 'datetime.timedelta'>
<class 'datetime.datetime'>


In [15]:
insights.log_likelihood

array([[      -inf],
       [2.21903657],
       [2.21903657],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [     